# Fibroblast cell-cycle response to injury, P1 vs P7/P8 (single-cell RNA-seq meta-analysis)
Result 3 (Figure 3). Kaggle notebook: **Internet ON, CPU**.

Data: GSE153480 (Wang Z 2020; MI vs sham at P1 and P8, days 1 and 3; one library per condition) and GSE247626 (Fan 2025; apical resection vs sham at P1 and P7, days 1, 4 and 7; non-myocytes). Five age-matched units.

Per unit: interaction log odds ratio = logOR(cycling | injury vs sham) at P7/P8 minus that at P1 (cycling = S or G2/M by `score_genes_cell_cycle`, Tirosh genes). Units pooled with DerSimonian-Laird random effects; pre-specified criterion: pooled estimate > 0, one-sided p < 0.05, >= 4 of 5 units positive.

This notebook is the fibroblast cell-cycle part of the Kaggle kernel that produced `results/result3_fibroblast_cycling_scrna.json`; cells of the same kernel that ran other pre-specified tests (listed in Supplementary Table S1) and the per-unit fibroblast/endothelial fraction were removed. The remaining code is unchanged apart from English comments, the output keys and file name.

In [ ]:
!pip install -q scanpy leidenalg igraph

In [ ]:
%%bash
set -e
mkdir -p f && cd f
G=https://ftp.ncbi.nlm.nih.gov/geo/series/GSE247nnn/GSE247626/suppl
for x in barcodes.tsv.gz cell_metadata.csv.gz features.tsv.gz processed_counts.mtx.gz; do wget -q $G/GSE247626_$x; done
mkdir -p r153 && cd r153
wget -q https://ftp.ncbi.nlm.nih.gov/geo/series/GSE153nnn/GSE153480/suppl/GSE153480_RAW.tar
tar xf GSE153480_RAW.tar && rm GSE153480_RAW.tar
ls -la

In [ ]:
import glob, json, os, numpy as np, pandas as pd, scanpy as sc, scipy.io as sio, scipy.sparse as sp, scipy.stats as ss
FB5 = "Pdgfra Tcf21 Dpt Gsn Smoc2".split()   # fibroblast cluster markers
EC4 = "Pecam1 Cdh5 Kdr Emcn".split()         # endothelial cluster markers
# Tirosh et al. S and G2/M genes (human symbols, converted to mouse capitalisation)
S_H = "MCM5 PCNA TYMS FEN1 MCM2 MCM4 RRM1 UNG GINS2 MCM6 CDCA7 DTL PRIM1 UHRF1 MLF1IP HELLS RFC2 RPA2 NASP RAD51AP1 GMNN WDR76 SLBP CCNE2 UBR7 POLD3 MSH2 ATAD2 RAD51 RRM2 CDC45 CDC6 EXO1 TIPIN DSCC1 BLM CASP8AP2 USP1 CLSPN POLA1 CHAF1B BRIP1 E2F8".split()
G2M_H = "HMGB2 CDK1 NUSAP1 UBE2C BIRC5 TPX2 TOP2A NDC80 CKS2 NUF2 CKS1B MKI67 TMPO CENPF TACC3 FAM64A SMC4 CCNB2 CKAP2L CKAP2 AURKB BUB1 KIF11 ANP32E TUBB4B GTSE1 KIF20B HJURP CDCA3 HN1 CDC20 TTK CDC25C KIF2C RANGAP1 NCAPD2 DLGAP5 CDCA2 CDCA8 ECT2 KIF23 HMMR AURKA PSRC1 ANLN LBR CKAP5 CENPE CTCF NEK2 G2E3 GAS2L3 CBX5 CENPA".split()
ms = lambda L: [g[0] + g[1:].lower() for g in L]
S_M, G2M_M = ms(S_H), ms(G2M_H)
res = {"note": "GSE130699 not used: all samples are PCM1+ cardiomyocyte nuclei (excluded in the analysis plan before any value was computed)"}

# ---------- GSE247626 (Fan 2025; P1 vs P7, apical resection vs sham, days 1/4/7; non-myocytes) ----------
X = sio.mmread("f/GSE247626_processed_counts.mtx.gz").tocsr()
bc = pd.read_csv("f/GSE247626_barcodes.tsv.gz", header=None, sep="\t")[0].astype(str).values
ft = pd.read_csv("f/GSE247626_features.tsv.gz", header=None, sep="\t")
genes = (ft[1] if ft.shape[1] > 1 else ft[0]).astype(str).values
if X.shape[0] == len(genes) and X.shape[1] == len(bc):
    X = X.T.tocsr()
meta = pd.read_csv("f/GSE247626_cell_metadata.csv.gz")
key = meta["barcode"].astype(str) if "barcode" in meta else meta.iloc[:, 0].astype(str)
pos = {b: i for i, b in enumerate(bc)}
idx = np.array([pos.get(b, -1) for b in key])
meta = meta[idx >= 0].reset_index(drop=True); X = X[idx[idx >= 0]]
print("is integer:", np.allclose(X.data[:10000], np.round(X.data[:10000])))
f = sc.AnnData(X=X.astype(np.float32)); f.var_names = genes; f.var_names_make_unique()
f.obs["sample"] = meta["orig.ident"].astype(str).values
f.obs["age"] = meta["time2"].astype(str).values; f.obs["cond"] = meta["time3"].map({"AR": "MI", "SH": "sham"}).astype(str).values
f.obs["day"] = meta["time1"].astype(str).str.replace("D", "").values; f.obs["auth"] = meta["celltype2"].astype(str).values

# ---------- GSE153480 (Wang Z 2020; P1 vs P8, MI vs sham, days 1/3; whole heart) ----------
ads = []
for m in sorted(glob.glob("f/r153/*_matrix.mtx.gz")):
    pre = m[: -len("matrix.mtx.gz")]
    s = os.path.basename(pre).split("_", 1)[1].rstrip("_")
    Xm = sio.mmread(m).T.tocsr().astype(np.float32)
    g = pd.read_csv(pre + "genes.tsv.gz", sep="\t", header=None)
    b = pd.read_csv(pre + "barcodes.tsv.gz", sep="\t", header=None)
    a = sc.AnnData(X=Xm); a.var_names = g[1].astype(str).values; a.var_names_make_unique()
    a.obs_names = [f"{s}:{x}" for x in b[0]]; a.obs["sample"] = s
    a.obs["age"] = s.split("_")[0]; a.obs["day"] = s.split("_")[1][0]; a.obs["cond"] = "MI" if "MI" in s else "sham"
    ads.append(a); print(s, a.shape)
w = sc.concat(ads, join="inner"); w.obs["auth"] = "NA"

def pipeline(a, mtmax, name):
    """QC, normalisation, clustering, fibroblast/endothelial cluster labels and cell-cycle phase."""
    a.var["mt"] = a.var_names.str.startswith("mt-")
    sc.pp.calculate_qc_metrics(a, qc_vars=["mt"], inplace=True)
    a = a[(a.obs.n_genes_by_counts >= 200) & (a.obs.pct_counts_mt < mtmax)].copy()
    raw = a.X.copy()
    sc.pp.normalize_total(a, 1e4); sc.pp.log1p(a)
    sc.pp.highly_variable_genes(a, n_top_genes=2000); sc.pp.pca(a, 30, random_state=0)
    sc.pp.neighbors(a, 15, random_state=0); sc.tl.leiden(a, 1.0, random_state=0)
    def pos2(gs):
        c = [a.var_names.get_loc(g) for g in gs if g in a.var_names]
        return (np.asarray((raw[:, c] > 0).sum(1)).ravel() >= 2)
    fb2, ec2 = pos2(FB5), pos2(EC4)
    lab = {}; info = {}
    for cl in a.obs.leiden.unique():
        mm = (a.obs.leiden == cl).values
        pf, pe = fb2[mm].mean(), ec2[mm].mean()
        t = "other"
        if pf >= 0.5 or pe >= 0.5:
            t = "FB" if pf >= pe else "EC"
        lab[cl] = t; info[cl] = {"n": int(mm.sum()), "fbfrac": float(pf), "ecfrac": float(pe), "type": t}
    a.obs["type"] = a.obs.leiden.map(lab).astype(str)
    sc.tl.score_genes_cell_cycle(a, s_genes=[g for g in S_M if g in a.var_names], g2m_genes=[g for g in G2M_M if g in a.var_names], random_state=0)
    a.obs["cyc"] = a.obs.phase.isin(["S", "G2M"]).values
    mk = [a.var_names.get_loc(g) for g in ("Mki67", "Top2a") if g in a.var_names]
    a.obs["cyc2"] = (np.asarray((raw[:, mk] > 0).sum(1)).ravel() >= 1)   # sensitivity: Mki67/Top2a expressed
    print(name, a.shape, pd.crosstab(a.obs["sample"], a.obs["type"]))
    return a, info

f, info247 = pipeline(f, 20, "GSE247626")
w, info153 = pipeline(w, 20, "GSE153480")
res["clusters"] = {"GSE247626": info247, "GSE153480": info153}

def lor(a1, b1, a0, b0):
    # a = cycling, b = not cycling; 1 = injury, 0 = sham; 0.5 continuity correction, Woolf variance
    a1, b1, a0, b0 = [x + 0.5 for x in (a1, b1, a0, b0)]
    return np.log(a1 * b0 / (b1 * a0)), 1 / a1 + 1 / b1 + 1 / a0 + 1 / b0
def unit(a, day, old, typekey="type", cyck="cyc"):
    """One age-matched injury/sham comparison: interaction logOR (older minus P1) for the fraction of cycling fibroblasts."""
    out = {}
    for age in ("P1", old):
        cnt = {}
        for c in ("MI", "sham"):
            mm = (a.obs.age == age) & (a.obs.day == day) & (a.obs.cond == c)
            fbm = mm & (a.obs[typekey] == "FB"); ecm = mm & (a.obs[typekey] == "EC")
            cnt[c] = {"fb": int(fbm.sum()), "ec": int(ecm.sum()), "fb_cyc": int((fbm & a.obs[cyck]).sum())}
        out[age] = cnt
    l_old, v_old = lor(out[old]["MI"]["fb_cyc"], out[old]["MI"]["fb"] - out[old]["MI"]["fb_cyc"], out[old]["sham"]["fb_cyc"], out[old]["sham"]["fb"] - out[old]["sham"]["fb_cyc"])
    l_p1, v_p1 = lor(out["P1"]["MI"]["fb_cyc"], out["P1"]["MI"]["fb"] - out["P1"]["MI"]["fb_cyc"], out["P1"]["sham"]["fb_cyc"], out["P1"]["sham"]["fb"] - out["P1"]["sham"]["fb_cyc"])
    return {"counts": out, "cyc_int": float(l_old - l_p1), "cyc_var": float(v_old + v_p1),
            "cyc_rate": {k: {c: (v[c]["fb_cyc"] / v[c]["fb"] if v[c]["fb"] else None) for c in v} for k, v in out.items()}}
def dl(est, var):
    """DerSimonian-Laird random-effects pooling."""
    est, var = np.array(est), np.array(var); wi = 1 / var
    fe = (wi * est).sum() / wi.sum(); Q = (wi * (est - fe) ** 2).sum(); k = len(est)
    tau2 = max(0, (Q - (k - 1)) / (wi.sum() - (wi ** 2).sum() / wi.sum())) if k > 1 else 0
    ws = 1 / (var + tau2); m = (ws * est).sum() / ws.sum(); se = np.sqrt(1 / ws.sum())
    return {"est": float(m), "se": float(se), "z": float(m / se), "p_one": float(ss.norm.sf(m / se)), "tau2": float(tau2), "Q": float(Q), "k": k}
def meta_run(units, ek, vk):
    e = [u[ek] for u in units.values()]; v = [u[vk] for u in units.values()]
    r = dl(e, v); r["n_pos"] = int(sum(x > 0 for x in e)); return r

def all_units(cyck="cyc", typekey247="type"):
    U = {}
    for d in ("1", "3"):
        U[f"GSE153480_d{d}"] = unit(w, d, "P8", "type", cyck)
    for d in ("1", "4", "7"):
        U[f"GSE247626_d{d}"] = unit(f, d, "P7", typekey247, cyck)
    return U
U = all_units()
m = meta_run(U, "cyc_int", "cyc_var")
k = m["k"]
m["verdict"] = "supported" if (m["est"] > 0 and m["p_one"] < 0.05 and m["n_pos"] >= int(np.ceil(0.7 * k))) else "not supported"
res["fibroblast_cycling_meta_analysis"] = {"units": U, "pooled": m, "rule": f"DL > 0, p < 0.05, n_pos >= ceil(0.7*{k})"}
# sensitivity 1: cycling defined by Mki67/Top2a expression
R = res["fibroblast_cycling_meta_analysis"]
U2 = all_units("cyc2"); R["sensitivity_Mki67_Top2a"] = {"meta": meta_run(U2, "cyc_int", "cyc_var"), "units": {k_: v["cyc_int"] for k_, v in U2.items()}}
# sensitivity 2: author-provided cell annotations (GSE247626 only)
f.obs["authtype"] = f.obs.auth.map(lambda x: x if x in ("FB", "EC") else "other")
U3 = {k_: unit(f, k_[-1], "P7", "authtype") for k_ in ("GSE247626_d1", "GSE247626_d4", "GSE247626_d7")}
R["sensitivity_author_annotation_GSE247626"] = {"meta": meta_run(U3, "cyc_int", "cyc_var"), "units": {k_: v["cyc_int"] for k_, v in U3.items()}}
# sensitivity 3: GSE247626 units only
U4 = {k_: v for k_, v in U.items() if k_.startswith("GSE247626")}
R["sensitivity_GSE247626_only"] = meta_run(U4, "cyc_int", "cyc_var")
json.dump(res, open("fibroblast_cycling.json", "w"), indent=1, default=float)
print(json.dumps(R["pooled"], indent=1, default=float))